# Alternative 2 — Transformer sur landmarks

**Objectif :** Déterminer si l'attention (mécanisme Transformer) apporte un gain par rapport au MLP lorsque les 42 articulations (2 mains × 21) sont traitées comme des tokens structurés.

Chaque articulation est un **token** de dimension 3 (x, y, z), augmenté d'embeddings
appris pour la position anatomique (21 articulations) et la main (gauche/droite). Un token [CLS]
agrège la séquence à travers un encodeur Transformer (2 couches, d_model=64, 4 têtes).
Les données sont des images statiques (pas de séquence temporelle) : on ne fabrique donc
PAS de séquence temporelle artificielle — l'attention opère sur la structure spatiale de la main.

**Protocole expérimental (commun à toutes les expériences) :**
- Dataset : 7 664 images du dataset W7CH (TSL), landmarks MediaPipe (21 points × 2 mains).
- Split stratifié 80/10/10 (seed 42) sur les échantillons ORIGINAUX, **avant**
  toute augmentation → aucune fuite de données entre train et validation/test.
- Augmentation (rotation ±20°, échelle 0.85-1.15, translation ±0.1, bruit 0.02)
  appliquée **uniquement** sur le train, équilibré à 200 échantillons/classe.
- Entraînement : Adam (lr 1e-3, wd 1e-4), CrossEntropy + label smoothing 0.1,
  ReduceLROnPlateau, early stopping sur val_acc, seeds fixes.
- 56 classes effectives sur 57 (les images `metro` ne sont pas détectées par
  MediaPipe), tête de classification à 57 sorties (comme le modèle actuel).


In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
ROOT = NB_DIR
while not (ROOT / "common").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for p in [str(ROOT), str(ROOT / "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

from common.paths import RESULTS_DIR, MODELS_DIR, PLOTS_DIR, REPO_ROOT
from common.data import load_dataset, normalized_inputs
from common.models import (
    count_params, set_seed, BaselineMLP, LandmarkTransformer, HandGCN, HandGAT,
)
from common.train import (
    Trainer, predict, full_metrics, save_metrics,
    plot_training_curves, plot_confusion, run_torch_benchmark,
)
from common import export as ex
from common.robustness import robustness_summary

set_seed(42)

def load_class_names():
    with open(REPO_ROOT / "data" / "processed" / "class_names.txt", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

class_names = load_class_names()
print("torch", torch.__version__)


torch 2.13.0+cpu


## 1. Chargement des données

Le dataset préparé (split + augmentation, cf. README) est chargé. Le modèle
reçoit 42 tokens (2 × 21 articulations) × 3 coordonnées + embeddings appris.


In [2]:
dataset = load_dataset()
X_train = torch.from_numpy(dataset["train"]["img_landmarks"]).float()
y_train = torch.from_numpy(dataset["train"]["labels"])
X_val   = torch.from_numpy(dataset["val"]["img_landmarks"]).float()
y_val   = torch.from_numpy(dataset["val"]["labels"])
X_test  = torch.from_numpy(dataset["test"]["img_landmarks"]).float()
y_test  = torch.from_numpy(dataset["test"]["labels"])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256, shuffle=False)
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape), "| test:", tuple(X_test.shape))


train: (13282, 2, 21, 3) | val: (767, 2, 21, 3) | test: (767, 2, 21, 3)


## 2. Architecture

Transformer encoder (norm_first) : 42 tokens (x,y,z) → Linear(3→64) + embedding
articulation + embedding main + token [CLS] + positionnel appris → 2 couches d'attention
(4 têtes, FFN 128) → tête 64 → 57. ~80k paramètres.

Nombre de paramètres : mesuré ci-dessous.


In [3]:
model = LandmarkTransformer(num_classes=57, d_model=64, nhead=4, num_layers=2, dim_ff=128)
print(f"params: {count_params(model):,}")


params: 79,481


## 3. Entraînement

Mêmes hyperparamètres que le protocole commun (Adam lr 1e-3, wd 1e-4,
label smoothing 0.1, early stopping patience 20, max 60 epochs).


In [4]:
trainer = Trainer(model, patience=20)
history = trainer.fit(train_loader, val_loader, max_epochs=60, early_stop=True)
trainer.restore_best()

plot_training_curves(history, PLOTS_DIR / "training_curves" / "transformer_curves.png",
                     title="Alternative 2 — Transformer sur landmarks")
print("best val acc:", trainer.best_val_acc)


  ep 1/60 train 4.0319/0.0352 val 3.9790/0.0522 (9.5s) best 0.0522


  ep 2/60 train 3.9827/0.0462 val 3.8904/0.0717 (9.6s) best 0.0717


  ep 3/60 train 3.7491/0.0757 val 3.5809/0.0978 (9.5s) best 0.0978


  ep 4/60 train 3.4396/0.1147 val 3.2952/0.1460 (9.6s) best 0.1460


  ep 5/60 train 3.2425/0.1629 val 3.0699/0.2021 (10.0s) best 0.2021


...

  ep 57/60 train 1.2643/0.8391 val 1.0950/0.9166 (11.1s) best 0.9244


  ep 58/60 train 1.2506/0.8448 val 1.0894/0.9140 (11.2s) best 0.9244


  ep 59/60 train 1.2444/0.8435 val 1.0767/0.9166 (11.2s) best 0.9244


  ep 60/60 train 1.2463/0.8460 val 1.0656/0.9231 (10.5s) best 0.9244


best val acc: 0.924380704041721


## 4. Évaluation sur le jeu de test

Métriques complètes : accuracy, precision/recall/F1 (macro et pondérés),
matrice de confusion, performance par classe.


In [5]:
y_pred, y_true = predict(model, test_loader)
metrics = full_metrics(y_true, y_pred, num_classes=57)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"macro F1: {metrics['macro_f1']:.4f} | weighted F1: {metrics['f1']:.4f}")
print(f"macro precision: {metrics['macro_precision']:.4f} | macro recall: {metrics['macro_recall']:.4f}")

cm = np.array(metrics["confusion_matrix"])
plot_confusion(cm, class_names, PLOTS_DIR / "confusion_matrices" / "transformer_cm.png",
               title="Alternative 2 — Transformer sur landmarks — confusion matrix (test)")
np.save(RESULTS_DIR / "confusion_transformer.npy", cm)

import pandas as pd
per = pd.DataFrame({
    "class": class_names,
    "precision": metrics["per_class_precision"],
    "recall": metrics["per_class_recall"],
    "f1": metrics["per_class_f1"],
    "support": metrics["per_class_support"],
})
per.to_csv(RESULTS_DIR / "per_class_transformer.csv", index=False)
print(per.sort_values("f1").head(8))


test accuracy: 0.9205
macro F1: 0.9196 | weighted F1: 0.9201
macro precision: 0.9178 | macro recall: 0.9323


     class  precision    recall        f1  support
54   metro   0.000000  0.000000  0.000000        0
39   jadda   0.642857  0.562500  0.600000       16
51     car   0.500000  0.833333  0.625000        6
52  karhba   0.500000  1.000000  0.666667        2
25   banka   0.785714  0.846154  0.814815       13
15   radio   0.941176  0.727273  0.820513       22
41     o5t   1.000000  0.724138  0.840000       29
17  siye7a   0.800000  0.888889  0.842105        9


## 5. Robustesse

Le modèle est évalué sous perturbations des landmarks appliquées **avant**
normalisation (dans l'espace image) : bruit gaussien, rotation, translation,
changement d'échelle — 5 essais par perturbation, moyenne ± écart-type.


In [6]:
def forward_fn(model, raw, batch_size=512):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(raw), batch_size):
            x = torch.from_numpy(raw[i:i + batch_size]).float()
            outs.append(model(x).argmax(1).numpy())
    return np.concatenate(outs)
rob = robustness_summary(model, dataset["test"]["img_landmarks"],
                         dataset["test"]["labels"], base_acc=metrics["accuracy"],
                         forward_fn=forward_fn)


  noise_0.01: acc=0.9132 (drop=+0.0073)


  noise_0.02: acc=0.8949 (drop=+0.0256)


  noise_0.05: acc=0.7562 (drop=+0.1643)


  rotation_5: acc=0.9100 (drop=+0.0104)


  rotation_10: acc=0.9022 (drop=+0.0183)


  rotation_20: acc=0.8357 (drop=+0.0847)


  translation_0.05: acc=0.9093 (drop=+0.0112)


  translation_0.10: acc=0.8699 (drop=+0.0506)


  scale_0.90: acc=0.9100 (drop=+0.0104)


  scale_1.10: acc=0.9218 (drop=-0.0013)


## 6. Mesures mobiles

- Nombre de paramètres et taille du checkpoint PyTorch.
- Export ONNX (FP32), vérification, latence ONNX Runtime CPU (batch=1).
- Versions FP16 et INT8 (quantification dynamique) : taille, latence,
  dégradation d'accuracy sur le jeu de test.


In [7]:
sample = X_test[:1].float()
lat_torch = run_torch_benchmark(model, sample)
print(f"torch latency: {lat_torch:.3f} ms/sample")

onnx_path = ex.export_onnx(model, sample, "transformer")
fp16 = ex.quantize_fp16(onnx_path, "transformer")
int8 = ex.quantize_int8(onnx_path, "transformer")

quantized = {}
for tag, p in [("fp32", onnx_path), ("fp16", fp16), ("int8", int8)]:
    preds = ex.ort_predict(p, X_test.numpy())
    quantized[tag] = {
        "size_mb": ex.file_size_mb(p),
        "latency_ms": ex.ort_latency(p, sample.numpy()),
        "test_acc": float((preds == y_test.numpy()).mean()),
    }
    print(f"  {tag}: size={quantized[tag]['size_mb']:.2f} MB "
          f"latency={quantized[tag]['latency_ms']:.3f} ms "
          f"acc={quantized[tag]['test_acc']:.4f}")

extra = {
    "robustness": rob,
    "epochs": trainer.epochs_run,
    "params": count_params(model),
    "best_val_acc": trainer.best_val_acc,
    "torch_latency_ms": lat_torch,
    "quantized": quantized,
}
save_metrics("transformer", metrics, extra=extra)

torch_path = ex.save_torch(model, "transformer")
print("saved:", torch_path, f"({ex.file_size_mb(torch_path):.2f} MB)")


torch latency: 0.634 ms/sample

  fp32: size=0.34 MB latency=0.193 ms acc=0.9205
  fp16: size=0.18 MB latency=0.190 ms acc=0.9205


  int8: size=0.15 MB latency=0.173 ms acc=0.9179
metrics saved -> D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\results\metrics_transformer.json
saved: D:\Stage\Tunisian-Sign-Language-Translator\alternative_models\models\transformer.pth (0.32 MB)


## 7. Conclusions

Question centrale : l'attention entre articulations apporte-t-elle quelque chose
de plus que les connexions implicites du MLP ? Avec un petit dataset statique, un Transformer
risque de sous-performer (moins d'inductive bias que le MLP/GCN, besoin de plus de données),
mais il peut capturer des relations globales entre doigts distants. Les résultats décideront.
